# PST-Parser su Colab: addestramento di r4 (analisi esplorativa R)

Notebook derivato da `notebooks/colab.ipynb` per l'addestramento del parser r4 (deviazione del 22/11/2026 in `protocol/deviations.md`). Stessa ricetta di r3 (`configs/experiments/baseline.yaml`, revisione del modello `a2856192…`); cambia solo il dataset: lo split di r3 (905/100/100) con i 168 + 30 task BigCodeBench aggiunti a train e validazione, gia preparato in `results/r4/train/`.

Differenze dal notebook originale:
- `RUN_ID` e `MODEL_REVISION` gia fissati;
- la configurazione aggiunge la sorgente `results/r4/data/bcb-r4.yaml` e legge record e split gia preparati: **non** si esegue `prepare-data`, che rimescolerebbe lo split di r3;
- tolti lo smoke test e le valutazioni sul test DevGPT (misure non dichiarate).

Caricare **solo** l'archivio sorgente indicato nelle istruzioni (ZIP LF di un HEAD committato). GPU di r3: NVIDIA A100-SXM4-80GB.


In [ ]:
import subprocess
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)


In [ ]:
from pathlib import Path
from google.colab import files
import hashlib
import os
import zipfile

uploaded = files.upload()  # selezionare un solo archivio sorgente ZIP
if len(uploaded) != 1:
    raise ValueError("Selezionare un solo archivio ZIP della sorgente")
archive_name, archive_bytes = next(iter(uploaded.items()))
SOURCE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
SOURCE_ROOT = Path("/content/pst-parser-source")
if SOURCE_ROOT.exists():
    raise FileExistsError("Sorgente gia estratta: riutilizzare il runtime solo dopo verifica esplicita")
SOURCE_ROOT.mkdir()
with zipfile.ZipFile(archive_name) as bundle:
    for member in bundle.infolist():
        target = (SOURCE_ROOT / member.filename).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()):
            raise ValueError("Archivio con percorso esterno alla directory sorgente")
        if Path(member.filename).name == ".env":
            raise ValueError("Rimuovere .env dall'archivio sorgente")
    bundle.extractall(SOURCE_ROOT)
roots = list(SOURCE_ROOT.rglob("pyproject.toml"))
if len(roots) != 1:
    raise ValueError("L'archivio deve contenere un solo progetto Python")
PROJECT = roots[0].parent
os.chdir(PROJECT)
print("source sha256:", SOURCE_SHA256)


In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.11.15"], check=True)
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11", "--extra", "cu128", "--extra", "train", "--extra", "unsloth", "--extra", "structured"], check=True)
UV_RUN = ["uv", "run", "--frozen", "--extra", "cu128", "--extra", "train", "--extra", "unsloth", "--extra", "structured"]
subprocess.run([*UV_RUN, "pstparser", "--version"], check=True)


In [ ]:
from google.colab import drive
import json
import re

drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"
RUN_ID = "parser-2026-11-22-r4"  # run r4: non riutilizzare per altre run
MODEL_REVISION = "a2856192dd7c25b842431f39c179a6c2c2f627d1"  # come r3
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID):
    raise ValueError("RUN_ID deve contenere solo lettere, cifre, _ e -")
if not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_REVISION):
    raise ValueError("Fissare una revisione concreta del modello prima di eseguire")
RUN_ROOT = Path("/content/drive/MyDrive/pst-parser-runs") / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = RUN_ROOT / "run-config.yaml"
IDENTITY = RUN_ROOT / "run-identity.json"
identity = {"run_id": RUN_ID, "source_sha256": SOURCE_SHA256, "model_revision": MODEL_REVISION}
if IDENTITY.exists():
    if json.loads(IDENTITY.read_text()) != identity:
        raise ValueError("RUN_ID gia associata a sorgente/modello diversi")
else:
    IDENTITY.write_text(json.dumps(identity, indent=2))


## Pipeline

Ogni comando usa la stessa configurazione risolta e termina la cella con un errore se lo stadio fallisce. I percorsi sono associati a RUN_ID; nessun adapter viene selezionato da una ricerca dell'ultima directory.

Per recuperare una sessione interrotta, ricarica lo stesso archivio e imposta la stessa RUN_ID e revisione modello. Verifica i manifest degli stadi gia completati e salta le relative celle. Una directory parziale senza manifest non e uno stadio completato: conservarla per diagnosi e utilizzare una nuova RUN_ID per una nuova esecuzione. Non viene promessa una ripresa automatica del training dal checkpoint.

Un microbatch diverso con accumulo compensato mantiene il batch effettivo, non garantisce identica traiettoria numerica. Eventuali override vanno stabiliti prima del freeze della configurazione.


In [ ]:
overrides = [
    f"name={json.dumps(RUN_ID)}",
    f"model.revision={json.dumps(MODEL_REVISION)}",
    "data.sources=[{path: data/raw/prompt_dataset_v2.xlsx, sheet: prompt_dataset_v2, fixes: data/raw/annotation_fixes.yaml}, {path: data/annotated/reasoning.yaml, format: yaml}, {path: results/r4/data/bcb-r4.yaml, format: yaml}]",
    "data.processed_dir=results/r4/train/prepared",
    "data.split.output_dir=results/r4/train/splits",
    f"training.output_dir={(RUN_ROOT / 'training').as_posix()}",
]
# Il progetto risolve e valida la configurazione nel proprio ambiente bloccato.
resolve = "import json,sys,yaml; from pstparser.config import load_experiment; c=load_experiment(sys.argv[1],overrides=json.loads(sys.argv[2])); print(yaml.safe_dump(c.model_dump(mode='json'),sort_keys=False))"
resolved = subprocess.run([*UV_RUN, "python", "-c", resolve, "configs/experiments/baseline.yaml", json.dumps(overrides)], check=True, capture_output=True, text=True).stdout
if CONFIG.exists():
    if CONFIG.read_text() != resolved:
        raise ValueError("Configurazione congelata diversa: utilizzare una nuova RUN_ID")
else:
    CONFIG.write_text(resolved)

def run(*arguments):
    subprocess.run([*UV_RUN, "pstparser", *map(str, arguments), "--config", str(CONFIG)], check=True)

# r4: record e split sono gia preparati nel sorgente (split di r3 conservato); niente prepare-data.
check = "from pathlib import Path; from pstparser.data.splits import load_split; s=load_split('results/r4/train/splits', records_path=Path('results/r4/train/prepared/records.jsonl')); print('split', len(s.train), len(s.val), len(s.test))"
subprocess.run([*UV_RUN, "python", "-c", check], check=True)  # atteso: split 1073 130 100


In [ ]:
TRAIN_DIR = RUN_ROOT / "training"
run("train", "--run-dir", TRAIN_DIR)


## Recupero degli artefatti

Gli artefatti sono gia su Drive sotto RUN_ID, anche se una cella successiva fallisce. Questa cella crea inoltre un archivio di trasporto con adapter finale, dataset preparato, split e relativi hash, configurazione risolta, lock, sorgenti Python, prompt e manifest di tutti gli stadi. I checkpoint intermedi sono esclusi; conservarli separatamente se servono per una futura ripresa del training. Nessuna credenziale e inclusa.


In [ ]:
import zipfile

ARCHIVE = RUN_ROOT / f"{RUN_ID}-artifacts.zip"
keep_names = {"predictions.jsonl", "results.json", "eval_details.jsonl", "alignments.jsonl", "training_metrics.jsonl", "run-config.yaml", "run-identity.json", "records.jsonl", "quality_report.json", "train.json", "val.json", "test.json", "generations.jsonl"}
with zipfile.ZipFile(ARCHIVE, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_ROOT.rglob("*")):
        if not path.is_file() or path == ARCHIVE:
            continue
        relative = path.relative_to(RUN_ROOT)
        if any(part.startswith("checkpoint-") for part in relative.parts):
            continue
        if path.name in keep_names or path.suffix == ".jsonl" or path.name.endswith("_manifest.json") or "adapter" in relative.parts:
            bundle.write(path, f"run/{relative.as_posix()}")
    for directory, pattern in (("src", "*.py"), ("configs", "*.yaml"), ("prompts", "*.md")):
        for path in sorted((PROJECT / directory).rglob(pattern)):
            bundle.write(path, f"source/{path.relative_to(PROJECT).as_posix()}")
    for name in ("uv.lock", "pyproject.toml", "README.md", "LICENSE"):
        bundle.write(PROJECT / name, f"source/{name}")
print("Artefatti conservati:", RUN_ROOT)
print("Archivio:", ARCHIVE)
